In [282]:
import sys
import os
import platform

print("Python version:", sys.version)
print("Operating system:", platform.platform())
print("Current directory:", os.getcwd())

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Operating system: Linux-6.18.55+-x86_64-with-glibc2.39
Current directory: /kaggle/working


In [283]:
import os

project_path = "/kaggle/working/ml-ops-copilot"

print("Project exists:", os.path.exists(project_path))
print("\nProject contents:")

for root, dirs, files in os.walk(project_path):
    level = root.replace(project_path, "").count(os.sep)
    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

Project exists: True

Project contents:
ml-ops-copilot/
    configs/
    tests/
    data/
        sample/
            lightgbm.md
    docs/
    app/
        main.py
        core/
            document_store.py
            __init__.py
            __pycache__/
                document_store.cpython-313.pyc
                __init__.cpython-313.pyc
        ingestion/
            document_loader.py
            text_chunker.py
            __init__.py
            __pycache__/
                document_loader.cpython-313.pyc
                __init__.cpython-313.pyc
                text_chunker.cpython-313.pyc
        retrieval/
        evaluation/
        generation/
        api/


In [284]:
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/ml-ops-copilot")

folders = [
    "app",
    "app/api",
    "app/core",
    "app/ingestion",
    "app/retrieval",
    "app/generation",
    "app/evaluation",
    "tests",
    "configs",
    "data/sample",
    "docs",
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project structure created successfully!")

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_dir():
        print("📁", path.relative_to(PROJECT_DIR))

Project structure created successfully!
📁 app
📁 app/api
📁 app/core
📁 app/core/__pycache__
📁 app/evaluation
📁 app/generation
📁 app/ingestion
📁 app/ingestion/__pycache__
📁 app/retrieval
📁 configs
📁 data
📁 data/sample
📁 docs
📁 tests


In [285]:
from pathlib import Path

main_file = PROJECT_DIR / "app" / "main.py"

main_file.write_text(
    '''
def main():
    print("ML Ops Copilot is starting...")


if __name__ == "__main__":
    main()
'''.strip(),
    encoding="utf-8"
)

print(main_file.read_text(encoding="utf-8"))

def main():
    print("ML Ops Copilot is starting...")


if __name__ == "__main__":
    main()


In [286]:
print(__name__)

__main__


In [287]:
def main():
    print("Application started")

print("Before condition")

if __name__ == "__main__":
    main()

print("After condition")

Before condition
Application started
After condition


In [288]:
class MLModel:

    def __init__(self):
        self.model = None

    def load_model(self):
        print("Loading model...")
        self.model = "Trained ML Model"

    def predict(self, data):
        if self.model is None:
            raise RuntimeError("Model is not loaded")

        return f"Prediction for {data}"

In [289]:
predictor = MLModel()

predictor.load_model()

result = predictor.predict([10, 20, 30])

print(result)

Loading model...
Prediction for [10, 20, 30]


In [290]:
class ModelRegistry:

    total_models = 0

    def __init__(self, model_name):
        self.model_name = model_name
        ModelRegistry.total_models += 1


m1 = ModelRegistry("LightGBM")
m2 = ModelRegistry("XGBoost")

print(m1.model_name)
print(m2.model_name)
print(ModelRegistry.total_models)

LightGBM
XGBoost
2


In [291]:
m1.total_models = 100

In [292]:
print(m1.total_models)
print(m2.total_models)
print(ModelRegistry.total_models)

100
2
2


In [293]:
%pip install -q fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [294]:
from fastapi import FastAPI

app = FastAPI(
    title="ML Ops Copilot",
    description="An AI assistant for Machine Learning Engineers",
    version="0.1.0"
)

@app.get("/")
def home():
    return {
        "message": "ML Ops Copilot is running",
        "version": "0.1.0"
    }

@app.get("/health")
def health_check():
    return {
        "status": "healthy"
    }

In [295]:
from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/")

print("Status Code:", response.status_code)
print("Response:", response.json())

response = client.get("/health")

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'message': 'ML Ops Copilot is running', 'version': '0.1.0'}
Status Code: 200
Response: {'status': 'healthy'}


In [296]:
from pydantic import BaseModel, Field


class DocumentCreate(BaseModel):
    title: str = Field(min_length=3, max_length=100)
    content: str = Field(min_length=10)
    source: str = "manual"

In [297]:
document = DocumentCreate(
    title="Fraud Detection",
    content="This document explains fraud detection using LightGBM."
)

print(document.model_dump())

{'title': 'Fraud Detection', 'content': 'This document explains fraud detection using LightGBM.', 'source': 'manual'}


In [298]:
from uuid import uuid4
from fastapi import HTTPException

# Temporary in-memory storage
documents = {}


@app.post("/documents", status_code=201)
def create_document(document: DocumentCreate):

    document_id = str(uuid4())

    documents[document_id] = {
        "id": document_id,
        **document.model_dump()
    }

    return {
        "message": "Document created successfully",
        "document": documents[document_id]
    }
    
    client = TestClient(app)

response = client.post(
    "/documents",
    json={
        "title": "Fraud Detection",
        "content": "This document explains fraud detection using LightGBM.",
        "source": "github"
    }
)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 201
Response: {'message': 'Document created successfully', 'document': {'id': '079551bf-5010-4b67-b9a0-468e3b3e1b2a', 'title': 'Fraud Detection', 'content': 'This document explains fraud detection using LightGBM.', 'source': 'github'}}


In [299]:
client = TestClient(app)

response = client.post(
    "/documents",
    json={
        "title": "Fraud Detection",
        "content": "This document explains fraud detection using LightGBM.",
        "source": "github"
    }
)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 201
Response: {'message': 'Document created successfully', 'document': {'id': '5c8bb7ab-e4c9-45e2-98d2-4c31127639d0', 'title': 'Fraud Detection', 'content': 'This document explains fraud detection using LightGBM.', 'source': 'github'}}


In [300]:
response = client.post(
    "/documents",
    json={
        "title": "AI",
        "content": "Short"
    }
)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 422
Response: {'detail': [{'type': 'string_too_short', 'loc': ['body', 'title'], 'msg': 'String should have at least 3 characters', 'input': 'AI', 'ctx': {'min_length': 3}}, {'type': 'string_too_short', 'loc': ['body', 'content'], 'msg': 'String should have at least 10 characters', 'input': 'Short', 'ctx': {'min_length': 10}}]}


In [301]:
@app.get("/documents/{document_id}")
def get_document(document_id: str):
    if document_id not in documents:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    return documents[document_id]

In [302]:
# First, create a document
create_response = client.post(
    "/documents",
    json={
        "title": "Fraud Detection",
        "content": "This document explains fraud detection using LightGBM.",
        "source": "github"
    }
)

document_id = create_response.json()["document"]["id"]

# Then, retrieve the document
get_response = client.get(f"/documents/{document_id}")

print("Status:", get_response.status_code)
print("Response:", get_response.json())

Status: 200
Response: {'id': 'c8442506-5c3a-4699-9472-36c74dd5ab40', 'title': 'Fraud Detection', 'content': 'This document explains fraud detection using LightGBM.', 'source': 'github'}


In [303]:
from fastapi import HTTPException

@app.get("/documents/{document_id}")
def get_document(document_id: str):
    if document_id not in documents:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    return documents[document_id]

In [304]:
create_response = client.post(
    "/documents",
    json={
        "title": "Fraud Detection",
        "content": "This document explains fraud detection using LightGBM.",
        "source": "github"
    }
)

document_id = create_response.json()["document"]["id"]

get_response = client.get(f"/documents/{document_id}")

print("Status:", get_response.status_code)
print("Response:", get_response.json())

Status: 200
Response: {'id': '47c5e9ba-aa2d-4fcf-99ff-4d2c99b7dea3', 'title': 'Fraud Detection', 'content': 'This document explains fraud detection using LightGBM.', 'source': 'github'}


In [305]:
response = client.get(
    "/documents/this-id-does-not-exist"
)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 404
Response: {'detail': 'Document not found'}


In [306]:
from pathlib import Path

project_path = Path("/kaggle/working/ml-ops-copilot")
core_path = project_path / "app" / "core"

core_path.mkdir(parents=True, exist_ok=True)

(core_path / "__init__.py").touch()

store_code = '''
from uuid import uuid4


class DocumentStore:
    def __init__(self):
        self.documents = {}

    def create(self, document_data):
        document_id = str(uuid4())

        document = {
            "id": document_id,
            **document_data
        }

        self.documents[document_id] = document
        return document

    def get(self, document_id):
        return self.documents.get(document_id)
'''

(core_path / "document_store.py").write_text(
    store_code,
    encoding="utf-8"
)

print("Document storage module created successfully!")

Document storage module created successfully!


In [307]:
import sys

project_path = "/kaggle/working/ml-ops-copilot"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from app.core.document_store import DocumentStore

document_store = DocumentStore()

print("DocumentStore initialized successfully!")

DocumentStore initialized successfully!


In [308]:
from app.core.document_store import DocumentStore

document_store = DocumentStore()

In [309]:
@app.post("/documents", status_code=201)
def create_document(document: DocumentCreate):

    saved_document = document_store.create(
        document.model_dump()
    )

    return {
        "message": "Document created successfully",
        "document": saved_document
    }

In [310]:
@app.get("/documents/{document_id}")
def get_document(document_id: str):

    document = document_store.get(document_id)

    if document is None:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    return document

In [311]:
from app.core.document_store import DocumentStore
from fastapi import HTTPException

# Create the storage instance
document_store = DocumentStore()


# Create document
@app.post("/documents", status_code=201)
def create_document(document: DocumentCreate):
    saved_document = document_store.create(
        document.model_dump()
    )

    return {
        "message": "Document created successfully",
        "document": saved_document
    }


# Retrieve document
@app.get("/documents/{document_id}")
def get_document(document_id: str):
    document = document_store.get(document_id)

    if document is None:
        raise HTTPException(
            status_code=404,
            detail="Document not found"
        )

    return document

In [312]:
# Create a document
create_response = client.post(
    "/documents",
    json={
        "title": "LightGBM Architecture",
        "content": "LightGBM uses gradient-based one-side sampling and exclusive feature bundling.",
        "source": "github"
    }
)

print("Create status:", create_response.status_code)

created_document = create_response.json()["document"]
document_id = created_document["id"]

# Retrieve the same document
get_response = client.get(
    f"/documents/{document_id}"
)

print("Get status:", get_response.status_code)
print("Retrieved document:", get_response.json())

Create status: 201
Get status: 200
Retrieved document: {'id': '3c3136fd-6f34-4f8e-86c4-20a972d1405b', 'title': 'LightGBM Architecture', 'content': 'LightGBM uses gradient-based one-side sampling and exclusive feature bundling.', 'source': 'github'}


In [313]:
%pip install -q pypdf

Note: you may need to restart the kernel to use updated packages.


In [314]:
from pathlib import Path

data_path = Path(
    "/kaggle/working/ml-ops-copilot/data/sample"
)

data_path.mkdir(parents=True, exist_ok=True)

print("Data directory:", data_path)

Data directory: /kaggle/working/ml-ops-copilot/data/sample


In [315]:
sample_text = """
# LightGBM Architecture

LightGBM is a gradient boosting framework.

It uses two important techniques:

1. Gradient-based One-Side Sampling (GOSS)
2. Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

EFB reduces the number of features by bundling mutually exclusive features.

LightGBM is designed for efficient training on large datasets.
"""

markdown_file = data_path / "lightgbm.md"

markdown_file.write_text(
    sample_text,
    encoding="utf-8"
)

print("Created:", markdown_file)

Created: /kaggle/working/ml-ops-copilot/data/sample/lightgbm.md


In [316]:
content = markdown_file.read_text(
    encoding="utf-8"
)

print(content)


# LightGBM Architecture

LightGBM is a gradient boosting framework.

It uses two important techniques:

1. Gradient-based One-Side Sampling (GOSS)
2. Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

EFB reduces the number of features by bundling mutually exclusive features.

LightGBM is designed for efficient training on large datasets.



In [317]:
from pathlib import Path

project_path = Path("/kaggle/working/ml-ops-copilot")
ingestion_path = project_path / "app" / "ingestion"

ingestion_path.mkdir(parents=True, exist_ok=True)

(ingestion_path / "__init__.py").touch()

In [318]:
loader_code = '''
from pathlib import Path
from pypdf import PdfReader


def load_document(file_path):
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}"
        )

    extension = path.suffix.lower()

    if extension in [".md", ".txt"]:
        text = path.read_text(encoding="utf-8")
        page_count = None

    elif extension == ".pdf":
        reader = PdfReader(str(path))

        pages = []

        for page_number, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text() or ""

            pages.append(
                f"\\n[Page {page_number}]\\n{page_text}"
            )

        text = "\\n".join(pages)
        page_count = len(reader.pages)

    else:
        raise ValueError(
            f"Unsupported file format: {extension}"
        )

    if not text.strip():
        raise ValueError("Document contains no extractable text")

    return {
        "filename": path.name,
        "file_type": extension,
        "content": text,
        "page_count": page_count
    }
'''

loader_file = ingestion_path / "document_loader.py"

loader_file.write_text(
    loader_code,
    encoding="utf-8"
)

print("Document Loader created:", loader_file)

Document Loader created: /kaggle/working/ml-ops-copilot/app/ingestion/document_loader.py


In [319]:
import sys

project_path = "/kaggle/working/ml-ops-copilot"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from app.ingestion.document_loader import load_document

document = load_document(
    "/kaggle/working/ml-ops-copilot/data/sample/lightgbm.md"
)

print("Filename:", document["filename"])
print("File type:", document["file_type"])
print("Content:")
print(document["content"])

Filename: lightgbm.md
File type: .md
Content:

# LightGBM Architecture

LightGBM is a gradient boosting framework.

It uses two important techniques:

1. Gradient-based One-Side Sampling (GOSS)
2. Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

EFB reduces the number of features by bundling mutually exclusive features.

LightGBM is designed for efficient training on large datasets.



In [320]:
from pathlib import Path

project_path = Path("/kaggle/working/ml-ops-copilot")

chunker_path = project_path / "app" / "ingestion" / "text_chunker.py"

chunker_code = '''
from uuid import uuid4


def chunk_text(
    text,
    document_id,
    filename,
    chunk_size=500,
    overlap=50
):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    words = text.split()

    chunks = []
    start = 0
    chunk_index = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        chunks.append({
            "chunk_id": str(uuid4()),
            "document_id": document_id,
            "chunk_index": chunk_index,
            "filename": filename,
            "text": chunk,
            "word_count": len(chunk.split())
        })

        chunk_index += 1

        if end >= len(words):
            break

        start = end - overlap

    return chunks
'''

chunker_path.write_text(chunker_code.strip() + "\n")

print("File created:", chunker_path.exists())
print("Location:", chunker_path)

File created: True
Location: /kaggle/working/ml-ops-copilot/app/ingestion/text_chunker.py


In [321]:
import sys

project_path = "/kaggle/working/ml-ops-copilot"

if project_path not in sys.path:
    sys.path.insert(0, project_path)

from app.ingestion.text_chunker import chunk_text

print("Text chunker imported successfully!")

Text chunker imported successfully!


In [322]:
from app.ingestion.text_chunker import chunk_text

chunks = chunk_text(
    text=document["content"],
    document_id="lightgbm-doc-001",
    filename=document["filename"],
    chunk_size=30,
    overlap=5
)

print("Number of chunks:", len(chunks))

for chunk in chunks:
    print("\n----------------")
    print("Chunk ID:", chunk["chunk_id"])
    print("Document ID:", chunk["document_id"])
    print("Chunk Index:", chunk["chunk_index"])
    print("Filename:", chunk["filename"])
    print("Word Count:", chunk["word_count"])
    print("Text:", chunk["text"])

Number of chunks: 2

----------------
Chunk ID: 7a3cda86-3406-4b10-a074-be6014e91259
Document ID: lightgbm-doc-001
Chunk Index: 0
Filename: lightgbm.md
Word Count: 30
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data

----------------
Chunk ID: ff1e4de5-9613-4f7f-b3a4-fc434663bc2e
Document ID: lightgbm-doc-001
Chunk Index: 1
Filename: lightgbm.md
Word Count: 29
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [323]:
from importlib import reload
import app.ingestion.text_chunker as chunker_module

reload(chunker_module)

chunks = chunker_module.chunk_text(
    text=document["content"],
    document_id="doc-001",
    filename=document["filename"],
    chunk_size=30,
    overlap=5
)

print("Number of chunks:", len(chunks))

for chunk in chunks:
    print("\n----------------")
    print("Chunk ID:", chunk["chunk_id"])
    print("Document ID:", chunk["document_id"])
    print("Chunk Index:", chunk["chunk_index"])
    print("Filename:", chunk["filename"])
    print("Word Count:", chunk["word_count"])
    print("Text:", chunk["text"])

Number of chunks: 2

----------------
Chunk ID: c761b9b1-ea76-4fbe-b93a-3c985ce6b5c9
Document ID: doc-001
Chunk Index: 0
Filename: lightgbm.md
Word Count: 30
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data

----------------
Chunk ID: 8f215931-6dd9-4526-94b0-f607329a65a3
Document ID: doc-001
Chunk Index: 1
Filename: lightgbm.md
Word Count: 29
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [324]:
%pip install -q sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [325]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [326]:
texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (2, 384)


In [327]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)

print(similarity)

[[1.0000001  0.77320427]
 [0.77320427 1.0000006 ]]


In [328]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def semantic_search(query, chunks, model, top_k=3):

    # Convert query into an embedding
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # Get document embeddings
    document_texts = [chunk["text"] for chunk in chunks]

    document_embeddings = model.encode(
        document_texts,
        normalize_embeddings=True
    )

    # Calculate similarity
    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # Sort results by similarity
    ranked_indices = np.argsort(similarities)[::-1]

    results = []

    for index in ranked_indices[:top_k]:
        results.append({
            "chunk": chunks[index],
            "score": float(similarities[index])
        })

    return results

In [329]:
query = "What is GOSS in LightGBM?"

results = semantic_search(
    query=query,
    chunks=chunks,
    model=model,
    top_k=2
)

for result in results:
    print("\nSimilarity:", result["score"])
    print("Source:", result["chunk"]["filename"])
    print("Chunk:", result["chunk"]["chunk_index"])
    print("Text:", result["chunk"]["text"])


Similarity: 0.6396982669830322
Source: lightgbm.md
Chunk: 0
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data

Similarity: 0.5376994013786316
Source: lightgbm.md
Chunk: 1
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [330]:
import numpy as np

def build_index(chunks, model):
    texts = [chunk["text"] for chunk in chunks]

    embeddings = model.encode(
        texts,
        normalize_embeddings=True
    )

    index = {
        "chunks": chunks,
        "embeddings": np.array(embeddings)
    }

    return index

In [331]:
document_index = build_index(chunks, model)

print("Number of chunks:", len(document_index["chunks"]))
print("Embedding shape:", document_index["embeddings"].shape)

Number of chunks: 2
Embedding shape: (2, 384)


In [332]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search(query, index, model, top_k=3):

    # فقط embedding سؤال را محاسبه می‌کنیم
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    # استفاده از embeddingهای از قبل محاسبه‌شده
    document_embeddings = index["embeddings"]

    # محاسبه شباهت
    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # مرتب‌سازی از بیشترین شباهت به کمترین
    ranked_indices = similarities.argsort()[::-1]

    results = []

    for idx in ranked_indices[:top_k]:
        results.append({
            "chunk": index["chunks"][idx],
            "score": float(similarities[idx])
        })

    return results

In [333]:
query = "What is GOSS in LightGBM?"

results = semantic_search(
    query=query,
    index=document_index,
    model=model,
    top_k=2
)

for result in results:
    print("\nSimilarity:", result["score"])
    print("Source:", result["chunk"]["filename"])
    print("Chunk:", result["chunk"]["chunk_index"])
    print("Text:", result["chunk"]["text"])


Similarity: 0.6396982669830322
Source: lightgbm.md
Chunk: 0
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data

Similarity: 0.5376994013786316
Source: lightgbm.md
Chunk: 1
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [334]:
!pip install -q qdrant-client

In [335]:
from importlib.metadata import version

print("Qdrant version:", version("qdrant-client"))

Qdrant version: 1.20.0


In [336]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

print("Qdrant client created successfully!")

Qdrant client created successfully!


In [337]:
from qdrant_client.models import Distance, VectorParams

collection_name = "ml_ops_documents"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print("Collection created successfully!")

Collection created successfully!


In [338]:
from qdrant_client.models import PointStruct
from uuid import uuid4

points = []

for chunk, embedding in zip(
    document_index["chunks"],
    document_index["embeddings"]
):
    point = PointStruct(
        id=str(uuid4()),
        vector=embedding.tolist(),
        payload={
            "document_id": chunk["document_id"],
            "filename": chunk["filename"],
            "chunk_index": chunk["chunk_index"],
            "text": chunk["text"],
            "word_count": chunk["word_count"]
        }
    )

    points.append(point)

print("Number of points:", len(points))

Number of points: 2


In [339]:
client.upsert(
    collection_name=collection_name,
    points=points
)

print("Points inserted successfully!")

Points inserted successfully!


In [340]:
collection_info = client.get_collection(
    collection_name=collection_name
)

print("Stored vectors:", collection_info.points_count)

Stored vectors: 2


In [341]:
query = "What is GOSS in LightGBM?"

query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)[0]

search_results = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    limit=2,
    with_payload=True
).points

for result in search_results:
    print("\nScore:", result.score)
    print("Filename:", result.payload["filename"])
    print("Chunk:", result.payload["chunk_index"])
    print("Text:", result.payload["text"])


Score: 0.6396982376053832
Filename: lightgbm.md
Chunk: 0
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data

Score: 0.5376994708759937
Filename: lightgbm.md
Chunk: 1
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [342]:
from qdrant_client.models import PointStruct
from uuid import uuid4

filtered_points = []

for chunk, embedding in zip(
    document_index["chunks"],
    document_index["embeddings"]
):
    point = PointStruct(
        id=str(uuid4()),
        vector=embedding.tolist(),
        payload={
            "document_id": chunk["document_id"],
            "filename": chunk["filename"],
            "chunk_index": chunk["chunk_index"],
            "text": chunk["text"],
            "word_count": chunk["word_count"],
            "category": "machine_learning"
        }
    )

    filtered_points.append(point)

print("Prepared points:", len(filtered_points))

Prepared points: 2


In [343]:
from qdrant_client.models import Distance, VectorParams

filtered_collection = "ml_ops_documents_filtered"

client.create_collection(
    collection_name=filtered_collection,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print("Filtered collection created successfully!")

Filtered collection created successfully!


In [344]:
client.upsert(
    collection_name="ml_ops_documents_filtered",
    points=filtered_points
)

print(
    "Stored points:",
    client.get_collection(
        "ml_ops_documents_filtered"
    ).points_count
)

Stored points: 2


In [345]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

search_results = client.query_points(
    collection_name="ml_ops_documents_filtered",
    query=query_embedding.tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value="machine_learning")
            )
        ]
    ),
    limit=2,
    with_payload=True
).points

for result in search_results:
    print("Score:", result.score)
    print("Category:", result.payload["category"])
    print("Filename:", result.payload["filename"])
    print("Text:", result.payload["text"])
    print("-" * 50)

Score: 0.6396982376053832
Category: machine_learning
Filename: lightgbm.md
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data
--------------------------------------------------
Score: 0.5376994708759937
Category: machine_learning
Filename: lightgbm.md
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.
--------------------------------------------------


In [346]:
docker_text = """
# Docker Fundamentals

Docker is a platform for containerization.

Containers package applications and their dependencies.

Docker images are used to create containers.

Docker helps make applications portable and reproducible.
"""

docker_chunks = chunk_text(
    text=docker_text,
    document_id="docker-doc-001",
    filename="docker.md",
    chunk_size=500,
    overlap=50
)

print("Docker chunks:", len(docker_chunks))
print("Content:", docker_chunks[0]["text"])

Docker chunks: 1
Content: # Docker Fundamentals Docker is a platform for containerization. Containers package applications and their dependencies. Docker images are used to create containers. Docker helps make applications portable and reproducible.


In [347]:
docker_embeddings = model.encode(
    [chunk["text"] for chunk in docker_chunks],
    normalize_embeddings=True
)

print("Embedding shape:", docker_embeddings.shape)

Embedding shape: (1, 384)


In [348]:
from qdrant_client.models import PointStruct
from uuid import uuid4

docker_point = PointStruct(
    id=str(uuid4()),
    vector=docker_embeddings[0].tolist(),
    payload={
        "document_id": "docker-doc-001",
        "filename": "docker.md",
        "chunk_index": 0,
        "text": docker_chunks[0]["text"],
        "category": "devops"
    }
)

client.upsert(
    collection_name="ml_ops_documents_filtered",
    points=[docker_point]
)

print(
    "Total points:",
    client.get_collection(
        "ml_ops_documents_filtered"
    ).points_count
)

Total points: 3


In [349]:
test_query = "How do containers package applications?"

test_embedding = model.encode(
    [test_query],
    normalize_embeddings=True
)[0]

unfiltered_results = client.query_points(
    collection_name="ml_ops_documents_filtered",
    query=test_embedding.tolist(),
    limit=3,
    with_payload=True
).points

for result in unfiltered_results:
    print(
        result.payload["filename"],
        "|",
        result.payload["category"],
        "| Score:",
        round(result.score, 4)
    )

docker.md | devops | Score: 0.5532
lightgbm.md | machine_learning | Score: 0.1067
lightgbm.md | machine_learning | Score: 0.0834


In [350]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

filtered_results = client.query_points(
    collection_name="ml_ops_documents_filtered",
    query=test_embedding.tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value="machine_learning")
            )
        ]
    ),
    limit=3,
    with_payload=True
).points

for result in filtered_results:
    print(
        result.payload["filename"],
        "|",
        result.payload["category"],
        "| Score:",
        round(result.score, 4)
    )

lightgbm.md | machine_learning | Score: 0.1067
lightgbm.md | machine_learning | Score: 0.0834


In [351]:
!pip install rank-bm25

In [352]:
from rank_bm25 import BM25Okapi

print("BM25 is ready!")

BM25 is ready!


In [353]:
# جمع‌آوری متن تمام Chunkها
all_chunks = (
    document_index["chunks"] + docker_chunks
)

# تبدیل متن‌ها به کلمات کوچک‌شده
tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in all_chunks
]

print("Number of documents:", len(tokenized_corpus))

for i, tokens in enumerate(tokenized_corpus):
    print(f"\nChunk {i}:")
    print(tokens[:15])

Number of documents: 3

Chunk 0:
['#', 'lightgbm', 'architecture', 'lightgbm', 'is', 'a', 'gradient', 'boosting', 'framework.', 'it', 'uses', 'two', 'important', 'techniques:', '1.']

Chunk 1:
['reduces', 'the', 'number', 'of', 'data', 'instances', 'used', 'during', 'training.', 'efb', 'reduces', 'the', 'number', 'of', 'features']

Chunk 2:
['#', 'docker', 'fundamentals', 'docker', 'is', 'a', 'platform', 'for', 'containerization.', 'containers', 'package', 'applications', 'and', 'their', 'dependencies.']


In [354]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")

BM25 index created successfully!


In [355]:
query = "What is GOSS in LightGBM?"

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)

ranked_indices = bm25_scores.argsort()[::-1]

for index in ranked_indices:
    print("Chunk:", index)
    print("Score:", round(bm25_scores[index], 4))
    print("Filename:", all_chunks[index]["filename"])
    print("Text:", all_chunks[index]["text"])
    print("-" * 50)

Chunk: 0
Score: 0.5752
Filename: lightgbm.md
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data
--------------------------------------------------
Chunk: 2
Score: 0.0706
Filename: docker.md
Text: # Docker Fundamentals Docker is a platform for containerization. Containers package applications and their dependencies. Docker images are used to create containers. Docker helps make applications portable and reproducible.
--------------------------------------------------
Chunk: 1
Score: 0.0706
Filename: lightgbm.md
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.
--------------------------------------------------


In [356]:
dense_results = client.query_points(
    collection_name="ml_ops_documents_filtered",
    query=model.encode(
        [query],
        normalize_embeddings=True
    )[0].tolist(),
    limit=3,
    with_payload=True
).points

dense_ranked = [
    {
        "filename": result.payload["filename"],
        "text": result.payload["text"],
        "score": result.score
    }
    for result in dense_results
]

for rank, result in enumerate(dense_ranked, start=1):
    print(
        f"Rank: {rank} | "
        f"File: {result['filename']} | "
        f"Score: {result['score']:.4f}"
    )

Rank: 1 | File: lightgbm.md | Score: 0.6397
Rank: 2 | File: lightgbm.md | Score: 0.5377
Rank: 3 | File: docker.md | Score: 0.0762


In [357]:
from collections import defaultdict

def reciprocal_rank_fusion(
    result_lists,
    k=60
):
    fused_scores = defaultdict(float)
    document_info = {}

    for results in result_lists:
        for result in results:
            key = result["key"]
            rank = result["rank"]

            fused_scores[key] += 1 / (k + rank)

            document_info[key] = {
                "filename": result["filename"],
                "text": result["text"]
            }

    ranked_results = []

    for key, score in fused_scores.items():
        ranked_results.append({
            "key": key,
            "filename": document_info[key]["filename"],
            "text": document_info[key]["text"],
            "rrf_score": score
        })

    ranked_results.sort(
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    return ranked_results

In [358]:
# آماده‌سازی رتبه‌بندی BM25
bm25_ranked = []

for rank, index in enumerate(ranked_indices, start=1):
    chunk = all_chunks[index]

    bm25_ranked.append({
        "key": (chunk["filename"], chunk["chunk_index"]),
        "filename": chunk["filename"],
        "text": chunk["text"],
        "rank": rank
    })


# آماده‌سازی رتبه‌بندی Dense
dense_ranked = []

for rank, result in enumerate(dense_results, start=1):
    dense_ranked.append({
        "key": (
            result.payload["filename"],
            result.payload["chunk_index"]
        ),
        "filename": result.payload["filename"],
        "text": result.payload["text"],
        "rank": rank
    })

print("BM25 documents:", len(bm25_ranked))
print("Dense documents:", len(dense_ranked))

BM25 documents: 3
Dense documents: 3


In [359]:
hybrid_results = reciprocal_rank_fusion(
    [bm25_ranked, dense_ranked]
)

for rank, result in enumerate(hybrid_results, start=1):
    print(
        f"Rank: {rank} | "
        f"File: {result['filename']} | "
        f"RRF Score: {result['rrf_score']:.5f}"
    )

Rank: 1 | File: lightgbm.md | RRF Score: 0.03279
Rank: 2 | File: docker.md | RRF Score: 0.03200
Rank: 3 | File: lightgbm.md | RRF Score: 0.03200


In [360]:
!pip install -q sentence-transformers

In [361]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512,
    device="cpu"
)

print("Cross-Encoder loaded successfully!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder loaded successfully!


In [362]:
rerank_query = "What is GOSS in LightGBM?"

rerank_candidates = [
    result["text"]
    for result in hybrid_results
]

pairs = [
    (rerank_query, document)
    for document in rerank_candidates
]

print("Number of pairs:", len(pairs))
print("\nFirst pair:")
print("Query:", pairs[0][0])
print("Document:", pairs[0][1])

Number of pairs: 3

First pair:
Query: What is GOSS in LightGBM?
Document: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data


In [363]:
# Calculate relevance scores using Cross-Encoder
reranker_scores = reranker.predict(pairs)

# Combine documents with their scores
reranked_results = []

for result, score in zip(hybrid_results, reranker_scores):
    reranked_results.append({
        "filename": result["filename"],
        "text": result["text"],
        "reranker_score": float(score)
    })

# Sort documents by relevance score
reranked_results.sort(
    key=lambda x: x["reranker_score"],
    reverse=True
)

# Display reranked results
for rank, result in enumerate(reranked_results, start=1):
    print(f"\nRank: {rank}")
    print("Filename:", result["filename"])
    print("Relevance Score:", round(result["reranker_score"], 4))
    print("Text:", result["text"][:200])


Rank: 1
Filename: lightgbm.md
Relevance Score: 7.2869
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the num

Rank: 2
Filename: lightgbm.md
Relevance Score: -2.0486
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.

Rank: 3
Filename: docker.md
Relevance Score: -11.3782
Text: # Docker Fundamentals Docker is a platform for containerization. Containers package applications and their dependencies. Docker images are used to create containers. Docker helps make applications por


In [364]:
def build_context(ranked_results, top_k=2):

    selected_results = ranked_results[:top_k]

    context_parts = []

    for rank, result in enumerate(selected_results, start=1):

        source = result["filename"]
        text = result["text"]

        context_part = f"""
[Source {rank}: {source}]

{text}
"""

        context_parts.append(context_part)

    context = "\n\n".join(context_parts)

    return context


context = build_context(
    ranked_results=reranked_results,
    top_k=2
)

print(context)


[Source 1: lightgbm.md]

# LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data



[Source 2: lightgbm.md]

reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.



In [365]:
def build_prompt(context, question):

    system_instructions = """
You are an ML engineering assistant.

Answer the user's question using only the provided context.

Do not introduce unsupported facts or assumptions.

If the context does not contain enough information,
clearly state that you do not have enough information.

Cite the source filename when providing factual information.
"""

    prompt = f"""
{system_instructions}

Context:
{context}

User Question:
{question}

Answer:
"""

    return prompt


question = "What is GOSS in LightGBM?"

prompt = build_prompt(
    context=context,
    question=question
)

print(prompt)



You are an ML engineering assistant.

Answer the user's question using only the provided context.

Do not introduce unsupported facts or assumptions.

If the context does not contain enough information,
clearly state that you do not have enough information.

Cite the source filename when providing factual information.


Context:

[Source 1: lightgbm.md]

# LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data



[Source 2: lightgbm.md]

reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


User Question:
What is GOSS in LightGBM?

Answer:



In [366]:
def check_evidence(ranked_results, min_score=0.0):

    if not ranked_results:
        return False

    best_score = ranked_results[0]["reranker_score"]

    return best_score >= min_score

In [367]:
evidence_available = check_evidence(
    ranked_results=reranked_results,
    min_score=0.0
)

print("Evidence available:", evidence_available)

Evidence available: True


In [368]:
unknown_question = "What is the mathematical formula of GOSS?"

unknown_pairs = [
    (unknown_question, result["text"])
    for result in hybrid_results
]

unknown_scores = reranker.predict(unknown_pairs)

unknown_ranked_results = []

for result, score in zip(hybrid_results, unknown_scores):
    unknown_ranked_results.append({
        "filename": result["filename"],
        "text": result["text"],
        "reranker_score": float(score)
    })

unknown_ranked_results.sort(
    key=lambda x: x["reranker_score"],
    reverse=True
)

print("Best relevance score:", unknown_ranked_results[0]["reranker_score"])

print(
    "Evidence available:",
    check_evidence(unknown_ranked_results, min_score=0.0)
)

Best relevance score: -1.249245524406433
Evidence available: False


In [369]:
def handle_abstention(evidence_available):
    if not evidence_available:
        return {
            "status": "insufficient_evidence",
            "answer": "I don't have enough information in the available documents to answer this question."
        }

    return {
        "status": "ready_for_generation",
        "answer": None
    }

In [370]:
result_with_evidence = handle_abstention(
    evidence_available=True
)

print(result_with_evidence)

{'status': 'ready_for_generation', 'answer': None}


In [371]:
result_without_evidence = handle_abstention(
    evidence_available=False
)

print(result_without_evidence)

{'status': 'insufficient_evidence', 'answer': "I don't have enough information in the available documents to answer this question."}


In [372]:
def make_generation_decision(ranked_results, min_score=0.0):
    evidence_available = check_evidence(
        ranked_results=ranked_results,
        min_score=min_score
    )

    decision = handle_abstention(
        evidence_available=evidence_available
    )

    return decision

In [373]:
supported_decision = make_generation_decision(
    ranked_results=reranked_results,
    min_score=0.0
)

print(supported_decision)

{'status': 'ready_for_generation', 'answer': None}


In [374]:
unsupported_decision = make_generation_decision(
    ranked_results=unknown_ranked_results,
    min_score=0.0
)

print(unsupported_decision)

{'status': 'insufficient_evidence', 'answer': "I don't have enough information in the available documents to answer this question."}


In [375]:
def generate_grounded_answer(prompt, evidence_available):
    if not evidence_available:
        return (
            "I don't have enough information "
            "in the available documents to answer this question."
        )

    return {
        "status": "ready_for_llm",
        "prompt": prompt
    }

In [376]:
generation_result = generate_grounded_answer(
    prompt=prompt,
    evidence_available=True
)

print("Status:", generation_result["status"])
print("\nPrompt:")
print(generation_result["prompt"])

Status: ready_for_llm

Prompt:


You are an ML engineering assistant.

Answer the user's question using only the provided context.

Do not introduce unsupported facts or assumptions.

If the context does not contain enough information,
clearly state that you do not have enough information.

Cite the source filename when providing factual information.


Context:

[Source 1: lightgbm.md]

# LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data



[Source 2: lightgbm.md]

reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


User Question:
What is GOSS in LightGBM?

Answer:



In [377]:
generation_result = generate_grounded_answer(
    prompt=prompt,
    evidence_available=True
)

print("Status:", generation_result["status"])
print("\nPrompt:")
print(generation_result["prompt"])

Status: ready_for_llm

Prompt:


You are an ML engineering assistant.

Answer the user's question using only the provided context.

Do not introduce unsupported facts or assumptions.

If the context does not contain enough information,
clearly state that you do not have enough information.

Cite the source filename when providing factual information.


Context:

[Source 1: lightgbm.md]

# LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data



[Source 2: lightgbm.md]

reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


User Question:
What is GOSS in LightGBM?

Answer:



In [378]:
def generate_extractive_answer(ranked_results, top_k=2):
    selected_results = ranked_results[:top_k]

    if not selected_results:
        return {
            "status": "insufficient_evidence",
            "answer": "I don't have enough information to answer this question."
        }

    answer_parts = []

    for result in selected_results:
        answer_parts.append({
            "text": result["text"],
            "source": result["filename"]
        })

    return {
        "status": "success",
        "answer": answer_parts
    }

In [379]:
extractive_result = generate_extractive_answer(
    ranked_results=reranked_results,
    top_k=2
)

print("Status:", extractive_result["status"])

for item in extractive_result["answer"]:
    print("\nSource:", item["source"])
    print("Text:", item["text"])

Status: success

Source: lightgbm.md
Text: # LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data

Source: lightgbm.md
Text: reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [380]:
unsupported_extractive_result = generate_extractive_answer(
    ranked_results=[],
    top_k=2
)

print(unsupported_extractive_result)

{'status': 'insufficient_evidence', 'answer': "I don't have enough information to answer this question."}


In [381]:
def run_rag_generation(ranked_results, min_score=0.0, top_k=2):

    decision = make_generation_decision(
        ranked_results=ranked_results,
        min_score=min_score
    )

    if decision["status"] != "ready_for_generation":
        return decision

    result = generate_extractive_answer(
        ranked_results=ranked_results,
        top_k=top_k
    )

    return result

In [382]:
rag_result = run_rag_generation(
    ranked_results=reranked_results,
    min_score=0.0,
    top_k=2
)

print("Status:", rag_result["status"])
print("\nAnswer:")

for item in rag_result["answer"]:
    print(f"\nSource: {item['source']}")
    print(item["text"])

Status: success

Answer:

Source: lightgbm.md
# LightGBM Architecture LightGBM is a gradient boosting framework. It uses two important techniques: 1. Gradient-based One-Side Sampling (GOSS) 2. Exclusive Feature Bundling (EFB) GOSS reduces the number of data

Source: lightgbm.md
reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [383]:
unsupported_rag_result = run_rag_generation(
    ranked_results=unknown_ranked_results,
    min_score=0.0,
    top_k=2
)

print(unsupported_rag_result)

{'status': 'insufficient_evidence', 'answer': "I don't have enough information in the available documents to answer this question."}


In [384]:
import re
from uuid import uuid4

def sentence_aware_chunking(
    text,
    document_id,
    filename,
    chunk_size=500,
    overlap=1
):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_sentences = []
    current_word_count = 0
    chunk_index = 0

    for sentence in sentences:
        sentence_word_count = len(sentence.split())

        if current_word_count + sentence_word_count > chunk_size and current_sentences:
            chunk_text = " ".join(current_sentences)

            chunks.append({
                "chunk_id": str(uuid4()),
                "document_id": document_id,
                "chunk_index": chunk_index,
                "filename": filename,
                "text": chunk_text,
                "word_count": len(chunk_text.split())
            })

            chunk_index += 1
            current_sentences = current_sentences[-overlap:] if overlap > 0 else []
            current_word_count = sum(
                len(s.split()) for s in current_sentences
            )

        current_sentences.append(sentence)
        current_word_count += sentence_word_count

    if current_sentences:
        chunk_text = " ".join(current_sentences)

        chunks.append({
            "chunk_id": str(uuid4()),
            "document_id": document_id,
            "chunk_index": chunk_index,
            "filename": filename,
            "text": chunk_text,
            "word_count": len(chunk_text.split())
        })

    return chunks

In [385]:
sentence_chunks = sentence_aware_chunking(
    text=document["content"],
    document_id="lightgbm-doc-001",
    filename=document["filename"],
    chunk_size=30,
    overlap=1
)

for chunk in sentence_chunks:
    print(f"\nChunk {chunk['chunk_index']}")
    print(f"Words: {chunk['word_count']}")
    print(chunk["text"])


Chunk 0
Words: 20
# LightGBM Architecture

LightGBM is a gradient boosting framework. It uses two important techniques:

1. Gradient-based One-Side Sampling (GOSS)
2.

Chunk 1
Words: 30
Gradient-based One-Side Sampling (GOSS)
2. Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training. EFB reduces the number of features by bundling mutually exclusive features.

Chunk 2
Words: 20
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [386]:
import re

def split_sentences(text):
    text = re.sub(r'(?m)^\s*\d+\.\s+', '', text)
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]

In [387]:
test_sentences = split_sentences(document["content"])

for index, sentence in enumerate(test_sentences, start=1):
    print(f"{index}. {sentence}")

1. # LightGBM Architecture

LightGBM is a gradient boosting framework.
2. It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.
3. EFB reduces the number of features by bundling mutually exclusive features.
4. LightGBM is designed for efficient training on large datasets.


In [388]:
def sentence_aware_chunking(
    text,
    document_id,
    filename,
    chunk_size=500,
    overlap=1
):
    sentences = split_sentences(text)

    chunks = []
    current_sentences = []
    current_word_count = 0
    chunk_index = 0

    for sentence in sentences:
        sentence_word_count = len(sentence.split())

        if (
            current_word_count + sentence_word_count > chunk_size
            and current_sentences
        ):
            chunk_text = " ".join(current_sentences)

            chunks.append({
                "chunk_id": str(uuid4()),
                "document_id": document_id,
                "chunk_index": chunk_index,
                "filename": filename,
                "text": chunk_text,
                "word_count": len(chunk_text.split())
            })

            chunk_index += 1

            current_sentences = (
                current_sentences[-overlap:]
                if overlap > 0 else []
            )

            current_word_count = sum(
                len(s.split()) for s in current_sentences
            )

        current_sentences.append(sentence)
        current_word_count += sentence_word_count

    if current_sentences:
        chunk_text = " ".join(current_sentences)

        chunks.append({
            "chunk_id": str(uuid4()),
            "document_id": document_id,
            "chunk_index": chunk_index,
            "filename": filename,
            "text": chunk_text,
            "word_count": len(chunk_text.split())
        })

    return chunks

In [389]:
sentence_chunks = sentence_aware_chunking(
    text=document["content"],
    document_id="lightgbm-doc-001",
    filename=document["filename"],
    chunk_size=30,
    overlap=0
)

for chunk in sentence_chunks:
    print(f"\nChunk {chunk['chunk_index']}")
    print(f"Words: {chunk['word_count']}")
    print(chunk["text"])


Chunk 0
Words: 9
# LightGBM Architecture

LightGBM is a gradient boosting framework.

Chunk 1
Words: 23
It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

Chunk 2
Words: 20
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [390]:
for chunk in sentence_chunks:
    print({
        "document_id": chunk["document_id"],
        "filename": chunk["filename"],
        "chunk_index": chunk["chunk_index"],
        "word_count": chunk["word_count"]
    })

{'document_id': 'lightgbm-doc-001', 'filename': 'lightgbm.md', 'chunk_index': 0, 'word_count': 9}
{'document_id': 'lightgbm-doc-001', 'filename': 'lightgbm.md', 'chunk_index': 1, 'word_count': 23}
{'document_id': 'lightgbm-doc-001', 'filename': 'lightgbm.md', 'chunk_index': 2, 'word_count': 20}


In [391]:
assert len(sentence_chunks) > 0

assert all(
    chunk["document_id"] == "lightgbm-doc-001"
    for chunk in sentence_chunks
)

assert all(
    chunk["chunk_index"] == index
    for index, chunk in enumerate(sentence_chunks)
)

assert all(
    chunk["word_count"] == len(chunk["text"].split())
    for chunk in sentence_chunks
)

print("Chunk metadata validation passed.")

Chunk metadata validation passed.


In [392]:
chunk_texts = [chunk["text"].strip() for chunk in sentence_chunks]

assert all(chunk_texts), "An empty chunk was found."

assert len(chunk_texts) == len(set(chunk_texts)), "Duplicate chunks were found."

print("Chunk content validation passed.")

Chunk content validation passed.


In [396]:
chunk_texts = [
    chunk["text"]
    for chunk in sentence_chunks
]

chunk_embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Number of chunks:", len(chunk_embeddings))
print("Embedding dimensions:", chunk_embeddings.shape)

Number of chunks: 3
Embedding dimensions: (3, 384)


In [399]:
print(chunk_embeddings[0])

[-6.31868392e-02 -6.44665658e-02 -2.63175983e-02  4.80616987e-02
  7.00154752e-02  3.06139719e-02  1.90066565e-02 -2.50861906e-02
 -8.85502473e-02 -4.28868607e-02  3.41662392e-02  1.61188804e-02
 -4.31190507e-04 -4.74744961e-02  5.55196963e-02  1.31654203e-01
  6.60633445e-02  3.01455576e-02 -4.83218171e-02 -5.68941645e-02
 -1.42108795e-04 -1.15071639e-01 -1.85830034e-02 -9.24746878e-03
 -3.28484401e-02  2.80558020e-02  8.34238902e-03 -2.59413704e-04
  7.45840967e-02 -1.79156959e-02  8.37982297e-02  4.61227112e-02
  5.35486639e-02  3.99915613e-02 -1.31006300e-01  1.82691198e-02
 -4.12669964e-03 -1.18629754e-01 -1.05108649e-01  1.74019746e-02
 -5.85867502e-02 -3.98496538e-02  3.49803455e-02  5.82734449e-03
  5.86444624e-02  1.88814867e-02  2.61671413e-02  1.63321663e-02
 -7.07922280e-02 -9.92003754e-02 -6.50232949e-04 -8.29392821e-02
 -4.85179611e-02  6.78258762e-02 -3.91486585e-02  7.93152452e-02
  4.05740328e-02 -1.51526881e-02  6.37602136e-02 -2.93001942e-02
 -1.76276769e-02 -9.36910

In [400]:
from pathlib import Path

project_root = Path("/kaggle/working/ml-ops-copilot")

for path in sorted(project_root.rglob("*")):
    if path.is_file():
        print(path.relative_to(project_root))

app/core/__init__.py
app/core/__pycache__/__init__.cpython-313.pyc
app/core/__pycache__/document_store.cpython-313.pyc
app/core/document_store.py
app/ingestion/__init__.py
app/ingestion/__pycache__/__init__.cpython-313.pyc
app/ingestion/__pycache__/document_loader.cpython-313.pyc
app/ingestion/__pycache__/text_chunker.cpython-313.pyc
app/ingestion/document_loader.py
app/ingestion/text_chunker.py
app/main.py
data/sample/lightgbm.md


In [401]:
from pathlib import Path

files = [
    "app/main.py",
    "app/core/document_store.py",
    "app/ingestion/document_loader.py",
    "app/ingestion/text_chunker.py",
]

for filename in files:
    path = project_root / filename
    print(f"\n{'=' * 60}")
    print(f"FILE: {filename}")
    print('=' * 60)
    print(path.read_text(encoding="utf-8"))


FILE: app/main.py
def main():
    print("ML Ops Copilot is starting...")


if __name__ == "__main__":
    main()

FILE: app/core/document_store.py

from uuid import uuid4


class DocumentStore:
    def __init__(self):
        self.documents = {}

    def create(self, document_data):
        document_id = str(uuid4())

        document = {
            "id": document_id,
            **document_data
        }

        self.documents[document_id] = document
        return document

    def get(self, document_id):
        return self.documents.get(document_id)


FILE: app/ingestion/document_loader.py

from pathlib import Path
from pypdf import PdfReader


def load_document(file_path):
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}"
        )

    extension = path.suffix.lower()

    if extension in [".md", ".txt"]:
        text = path.read_text(encoding="utf-8")
        page_count = None

    elif extension == ".p

In [403]:
from pathlib import Path

file_path = Path(
    "/kaggle/working/ml-ops-copilot/app/ingestion/text_chunker.py"
)

code = '''import re
from uuid import uuid4


def split_sentences(text):
    text = re.sub(r'(?m)^\\s*\\d+\\.\\s+', '', text)
    sentences = re.split(r'(?<=[.!?])\\s+', text.strip())

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def sentence_aware_chunking(
    text,
    document_id,
    filename,
    chunk_size=500,
    overlap=0
):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    sentences = split_sentences(text)

    chunks = []
    current_sentences = []
    current_word_count = 0
    chunk_index = 0

    for sentence in sentences:
        sentence_word_count = len(sentence.split())

        if (
            current_word_count + sentence_word_count > chunk_size
            and current_sentences
        ):
            chunk_text = " ".join(current_sentences)

            chunks.append({
                "chunk_id": str(uuid4()),
                "document_id": document_id,
                "chunk_index": chunk_index,
                "filename": filename,
                "text": chunk_text,
                "word_count": len(chunk_text.split())
            })

            chunk_index += 1

            current_sentences = (
                current_sentences[-overlap:]
                if overlap > 0 else []
            )

            current_word_count = sum(
                len(sentence.split())
                for sentence in current_sentences
            )

        current_sentences.append(sentence)
        current_word_count += sentence_word_count

    if current_sentences:
        chunk_text = " ".join(current_sentences)

        chunks.append({
            "chunk_id": str(uuid4()),
            "document_id": document_id,
            "chunk_index": chunk_index,
            "filename": filename,
            "text": chunk_text,
            "word_count": len(chunk_text.split())
        })

    return chunks
'''

file_path.write_text(code, encoding="utf-8")

print("Updated:", file_path)
print("File size:", file_path.stat().st_size, "bytes")

Updated: /kaggle/working/ml-ops-copilot/app/ingestion/text_chunker.py
File size: 1997 bytes


In [407]:
import importlib
import app.ingestion.text_chunker as text_chunker

# Reload the module from the updated file
importlib.reload(text_chunker)

# Check whether the function exists
print("Available functions:", [
    name for name in dir(text_chunker)
    if not name.startswith("_")
])

print(
    "Function found:",
    hasattr(text_chunker, "sentence_aware_chunking")
)

Available functions: ['chunk_text', 're', 'sentence_aware_chunking', 'split_sentences', 'uuid4']
Function found: True


In [408]:
from pathlib import Path
from app.ingestion.text_chunker import sentence_aware_chunking

document_path = Path(
    "/kaggle/working/ml-ops-copilot/data/sample/lightgbm.md"
)

text = document_path.read_text(encoding="utf-8")

chunks = sentence_aware_chunking(
    text=text,
    document_id="lightgbm-doc-001",
    filename=document_path.name,
    chunk_size=30,
    overlap=0
)

for chunk in chunks:
    print(f"\nChunk {chunk['chunk_index']}")
    print("Words:", chunk["word_count"])
    print("Text:", chunk["text"])


Chunk 0
Words: 9
Text: # LightGBM Architecture

LightGBM is a gradient boosting framework.

Chunk 1
Words: 23
Text: It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

Chunk 2
Words: 20
Text: EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [409]:
from pathlib import Path

retrieval_dir = Path(
    "/kaggle/working/ml-ops-copilot/app/retrieval"
)

retrieval_dir.mkdir(parents=True, exist_ok=True)

(retrieval_dir / "__init__.py").touch()

embedding_code = '''from sentence_transformers import SentenceTransformer


class EmbeddingModel:
    def __init__(
        self,
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    ):
        self.model = SentenceTransformer(model_name)

    def encode(self, texts):
        return self.model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=False
        )
'''

(retrieval_dir / "embeddings.py").write_text(
    embedding_code,
    encoding="utf-8"
)

print("Created:", retrieval_dir / "embeddings.py")

Created: /kaggle/working/ml-ops-copilot/app/retrieval/embeddings.py


In [410]:
from pathlib import Path

embedding_file = Path(
    "/kaggle/working/ml-ops-copilot/app/retrieval/embeddings.py"
)

embedding_code = '''from sentence_transformers import SentenceTransformer


class EmbeddingModel:
    def __init__(
        self,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model=None
    ):
        if model is not None:
            self.model = model
        else:
            self.model = SentenceTransformer(model_name)

    def encode(self, texts):
        return self.model.encode(
            texts,
            convert_to_numpy=True,
            show_progress_bar=False
        )
'''

embedding_file.write_text(embedding_code, encoding="utf-8")

print("Embedding module updated.")

Embedding module updated.


In [411]:
from app.retrieval.embeddings import EmbeddingModel

# Reuse the already loaded SentenceTransformer model
embedding_model = EmbeddingModel(model=model)

# Extract text from the chunks
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate embeddings
embeddings = embedding_model.encode(chunk_texts)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", embeddings.shape)

Number of chunks: 3
Embedding shape: (3, 384)


In [412]:
from pathlib import Path

vector_store_code = '''from uuid import uuid4

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams


class VectorStore:
    def __init__(
        self,
        collection_name="ml_ops_documents",
        vector_size=384
    ):
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name

        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(
                size=vector_size,
                distance=Distance.COSINE
            )
        )

    def add_chunks(self, chunks, embeddings):
        if len(chunks) != len(embeddings):
            raise ValueError(
                "The number of chunks must match the number of embeddings"
            )

        points = []

        for chunk, embedding in zip(chunks, embeddings):
            points.append(
                PointStruct(
                    id=str(uuid4()),
                    vector=embedding.tolist(),
                    payload={
                        "document_id": chunk["document_id"],
                        "filename": chunk["filename"],
                        "chunk_index": chunk["chunk_index"],
                        "text": chunk["text"],
                        "word_count": chunk["word_count"]
                    }
                )
            )

        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )

        return len(points)

    def search(self, query_embedding, top_k=3):
        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_embedding.tolist(),
            limit=top_k,
            with_payload=True
        ).points

        return [
            {
                "chunk": result.payload,
                "score": float(result.score)
            }
            for result in results
        ]
'''

project_root = Path("/kaggle/working/ml-ops-copilot")
file_path = project_root / "app/retrieval/vector_store.py"

file_path.write_text(vector_store_code, encoding="utf-8")

print("Created:", file_path)

Created: /kaggle/working/ml-ops-copilot/app/retrieval/vector_store.py


In [413]:
from app.retrieval.vector_store import VectorStore

# Create the vector store
vector_store = VectorStore()

# Store chunks and their embeddings
stored_count = vector_store.add_chunks(chunks, embeddings)

print("Stored chunks:", stored_count)

# Encode a search query
query = "What techniques does LightGBM use?"
query_embedding = embedding_model.encode([query])[0]

# Search for relevant chunks
results = vector_store.search(
    query_embedding=query_embedding,
    top_k=3
)

# Display search results
for result in results:
    print("\nScore:", round(result["score"], 4))
    print("Filename:", result["chunk"]["filename"])
    print("Text:", result["chunk"]["text"])

Stored chunks: 3

Score: 0.6809
Filename: lightgbm.md
Text: # LightGBM Architecture

LightGBM is a gradient boosting framework.

Score: 0.6288
Filename: lightgbm.md
Text: EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.

Score: 0.3628
Filename: lightgbm.md
Text: It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.


In [414]:
from pathlib import Path

bm25_code = '''import re

from rank_bm25 import BM25Okapi


class BM25Retriever:
    def __init__(self, chunks):
        self.chunks = chunks

        self.tokenized_corpus = [
            self._tokenize(chunk["text"])
            for chunk in chunks
        ]

        self.bm25 = BM25Okapi(self.tokenized_corpus)

    @staticmethod
    def _tokenize(text):
        return re.findall(r"\\b\\w+\\b", text.lower())

    def search(self, query, top_k=3):
        tokenized_query = self._tokenize(query)

        scores = self.bm25.get_scores(tokenized_query)

        ranked_indices = sorted(
            range(len(scores)),
            key=lambda index: scores[index],
            reverse=True
        )[:top_k]

        return [
            {
                "chunk": self.chunks[index],
                "score": float(scores[index])
            }
            for index in ranked_indices
        ]
'''

file_path = Path(
    "/kaggle/working/ml-ops-copilot/app/retrieval/bm25.py"
)

file_path.write_text(bm25_code, encoding="utf-8")

print("Created:", file_path)

Created: /kaggle/working/ml-ops-copilot/app/retrieval/bm25.py


In [415]:
from pathlib import Path
import importlib

bm25_code = r'''import re

from rank_bm25 import BM25Okapi


class BM25Retriever:
    def __init__(self, chunks):
        self.chunks = chunks

        self.tokenized_corpus = [
            self._tokenize(chunk["text"])
            for chunk in chunks
        ]

        self.bm25 = BM25Okapi(self.tokenized_corpus)

    @staticmethod
    def _tokenize(text):
        return re.findall(r"\b\w+\b", text.lower())

    def search(self, query, top_k=3):
        tokenized_query = self._tokenize(query)

        scores = self.bm25.get_scores(tokenized_query)

        ranked_indices = sorted(
            range(len(scores)),
            key=lambda index: scores[index],
            reverse=True
        )[:top_k]

        return [
            {
                "chunk": self.chunks[index],
                "score": float(scores[index])
            }
            for index in ranked_indices
        ]
'''

file_path = Path(
    "/kaggle/working/ml-ops-copilot/app/retrieval/bm25.py"
)

file_path.write_text(bm25_code, encoding="utf-8")

import app.retrieval.bm25 as bm25_module
importlib.reload(bm25_module)

bm25_retriever = bm25_module.BM25Retriever(chunks)

bm25_results = bm25_retriever.search(
    "What techniques does LightGBM use?",
    top_k=3
)

for result in bm25_results:
    print("\nScore:", round(result["score"], 4))
    print("Text:", result["chunk"]["text"])


Score: 0.4304
Text: It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

Score: 0.0965
Text: # LightGBM Architecture

LightGBM is a gradient boosting framework.

Score: 0.0525
Text: EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.


In [416]:
from pathlib import Path

hybrid_code = '''class HybridRetriever:
    def __init__(self, vector_store, bm25_retriever, rrf_k=60):
        self.vector_store = vector_store
        self.bm25_retriever = bm25_retriever
        self.rrf_k = rrf_k

    def search(self, query, query_embedding, top_k=3):
        dense_results = self.vector_store.search(
            query_embedding=query_embedding,
            top_k=top_k
        )

        sparse_results = self.bm25_retriever.search(
            query=query,
            top_k=top_k
        )

        scores = {}
        chunks = {}

        for results in [dense_results, sparse_results]:
            for rank, result in enumerate(results, start=1):
                chunk = result["chunk"]
                chunk_id = chunk["chunk_id"]

                scores[chunk_id] = (
                    scores.get(chunk_id, 0)
                    + 1 / (self.rrf_k + rank)
                )

                chunks[chunk_id] = chunk

        ranked_results = sorted(
            [
                {
                    "chunk": chunks[chunk_id],
                    "score": score
                }
                for chunk_id, score in scores.items()
            ],
            key=lambda item: item["score"],
            reverse=True
        )

        return ranked_results[:top_k]
'''

file_path = Path(
    "/kaggle/working/ml-ops-copilot/app/retrieval/hybrid_search.py"
)

file_path.write_text(hybrid_code, encoding="utf-8")

print("Created:", file_path)

Created: /kaggle/working/ml-ops-copilot/app/retrieval/hybrid_search.py


In [420]:
from pathlib import Path
import importlib

# Define the project directory
project_dir = Path("/kaggle/working/ml-ops-copilot")

# Define the VectorStore implementation
vector_store_code = r'''
from uuid import uuid4

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams


class VectorStore:
    def __init__(self, collection_name="ml_ops_documents", vector_size=384):
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name

        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(
                size=vector_size,
                distance=Distance.COSINE
            )
        )

    def add_chunks(self, chunks, embeddings):
        if len(chunks) != len(embeddings):
            raise ValueError(
                "The number of chunks must match the number of embeddings"
            )

        points = []

        for chunk, embedding in zip(chunks, embeddings):
            points.append(
                PointStruct(
                    id=str(uuid4()),
                    vector=embedding.tolist(),
                    payload={
                        "chunk_id": chunk["chunk_id"],
                        "document_id": chunk["document_id"],
                        "filename": chunk["filename"],
                        "chunk_index": chunk["chunk_index"],
                        "text": chunk["text"],
                        "word_count": chunk["word_count"]
                    }
                )
            )

        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )

        return len(points)

    def search(self, query_embedding, top_k=3):
        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_embedding.tolist(),
            limit=top_k,
            with_payload=True
        ).points

        return [
            {
                "chunk": result.payload,
                "score": float(result.score)
            }
            for result in results
        ]
'''

# Write the updated file
vector_store_path = project_dir / "app/retrieval/vector_store.py"

vector_store_path.write_text(
    vector_store_code,
    encoding="utf-8"
)

print("Updated:", vector_store_path)

# Reload the updated VectorStore module
import app.retrieval.vector_store as vector_store_module
importlib.reload(vector_store_module)

# Create a fresh in-memory vector database
vector_store = vector_store_module.VectorStore()

# Store document chunks and their embeddings
stored_count = vector_store.add_chunks(chunks, embeddings)

# Reload the HybridRetriever module
import app.retrieval.hybrid_search as hybrid_module
importlib.reload(hybrid_module)

# Initialize HybridRetriever
hybrid_retriever = hybrid_module.HybridRetriever(
    vector_store=vector_store,
    bm25_retriever=bm25_retriever
)

# Prepare the query
query = "What techniques does LightGBM use?"
query_embedding = embedding_model.encode([query])[0]

# Execute hybrid retrieval
hybrid_results = hybrid_retriever.search(
    query=query,
    query_embedding=query_embedding,
    top_k=3
)

# Display results
print("\nStored chunks:", stored_count)
print("\nHybrid Search Results:")

for result in hybrid_results:
    print(
        f"Score: {result['score']:.5f} | "
        f"Chunk: {result['chunk']['chunk_index']} | "
        f"File: {result['chunk']['filename']}"
    )
    print(result["chunk"]["text"])
    print("-" * 60)

Updated: /kaggle/working/ml-ops-copilot/app/retrieval/vector_store.py

Stored chunks: 3

Hybrid Search Results:
Score: 0.03252 | Chunk: 0 | File: lightgbm.md
# LightGBM Architecture

LightGBM is a gradient boosting framework.
------------------------------------------------------------
Score: 0.03227 | Chunk: 1 | File: lightgbm.md
It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.
------------------------------------------------------------
Score: 0.03200 | Chunk: 2 | File: lightgbm.md
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.
------------------------------------------------------------


In [421]:
from pathlib import Path

reranker_code = r'''
from sentence_transformers import CrossEncoder


class CrossEncoderReranker:
    def __init__(
        self,
        model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        model=None
    ):
        self.model = model or CrossEncoder(model_name)

    def rerank(self, query, results, top_k=3):
        if not results:
            return []

        pairs = [
            (query, result["chunk"]["text"])
            for result in results
        ]

        scores = self.model.predict(pairs)

        reranked_results = []

        for result, score in zip(results, scores):
            reranked_results.append({
                "chunk": result["chunk"],
                "score": float(score)
            })

        reranked_results.sort(
            key=lambda item: item["score"],
            reverse=True
        )

        return reranked_results[:top_k]
'''

reranker_path = (
    Path("/kaggle/working/ml-ops-copilot")
    / "app/retrieval/reranker.py"
)

reranker_path.write_text(reranker_code, encoding="utf-8")

print("Created:", reranker_path)

Created: /kaggle/working/ml-ops-copilot/app/retrieval/reranker.py


In [423]:
import importlib
import app.retrieval.reranker as reranker_module

importlib.reload(reranker_module)

# Load the Cross-Encoder model
reranker = reranker_module.CrossEncoderReranker()

print("Cross-Encoder model loaded.")

# Rerank hybrid retrieval results
reranked_results = reranker.rerank(
    query=query,
    results=hybrid_results,
    top_k=3
)

# Display reranked results
print("\nReranked Results:\n")

for result in reranked_results:
    print(
        f"Relevance Score: {result['score']:.4f} | "
        f"Chunk: {result['chunk']['chunk_index']} | "
        f"File: {result['chunk']['filename']}"
    )
    print(result["chunk"]["text"])
    print("-" * 60)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-Encoder model loaded.

Reranked Results:

Relevance Score: 2.4257 | Chunk: 2 | File: lightgbm.md
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.
------------------------------------------------------------
Relevance Score: 2.1311 | Chunk: 0 | File: lightgbm.md
# LightGBM Architecture

LightGBM is a gradient boosting framework.
------------------------------------------------------------
Relevance Score: -2.9119 | Chunk: 1 | File: lightgbm.md
It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.
------------------------------------------------------------


In [424]:
from pathlib import Path

context_builder_code = r'''
def build_context(ranked_results, top_k=3):
    selected_results = ranked_results[:top_k]

    context_parts = []

    for index, result in enumerate(selected_results, start=1):
        chunk = result["chunk"]

        source = (
            f"[Source {index}: {chunk['filename']}]\n"
            f"{chunk['text']}"
        )

        context_parts.append(source)

    return "\n\n".join(context_parts)
'''

project_dir = Path("/kaggle/working/ml-ops-copilot")
generation_dir = project_dir / "app/generation"

generation_dir.mkdir(parents=True, exist_ok=True)

context_path = generation_dir / "context_builder.py"

context_path.write_text(
    context_builder_code,
    encoding="utf-8"
)

print("Created:", context_path)

Created: /kaggle/working/ml-ops-copilot/app/generation/context_builder.py


In [425]:
from pathlib import Path

prompt_builder_code = r'''
def build_prompt(query, context):
    return f"""
You are an ML engineering assistant.

Answer the user's question using only the provided context.

Rules:
1. Do not introduce unsupported facts or assumptions.
2. If the context does not contain enough information,
   clearly state that you do not have enough information.
3. Cite the source filename when providing factual claims.
4. Do not invent citations.

Context:
{context}

Question:
{query}

Answer:
"""
'''

project_dir = Path("/kaggle/working/ml-ops-copilot")
prompt_path = project_dir / "app/generation/prompt_builder.py"

prompt_path.write_text(
    prompt_builder_code,
    encoding="utf-8"
)

print("Created:", prompt_path)

Created: /kaggle/working/ml-ops-copilot/app/generation/prompt_builder.py


In [426]:
import importlib

import app.generation.context_builder as context_module
import app.generation.prompt_builder as prompt_module

importlib.reload(context_module)
importlib.reload(prompt_module)

# Build context from reranked results
context = context_module.build_context(
    ranked_results=reranked_results,
    top_k=3
)

# Build the final prompt
prompt = prompt_module.build_prompt(
    query=query,
    context=context
)

print("Generated Context:\n")
print(context)

print("\n" + "=" * 70)
print("Generated Prompt:\n")
print(prompt)

Generated Context:

[Source 1: lightgbm.md]
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.

[Source 2: lightgbm.md]
# LightGBM Architecture

LightGBM is a gradient boosting framework.

[Source 3: lightgbm.md]
It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

Generated Prompt:


You are an ML engineering assistant.

Answer the user's question using only the provided context.

Rules:
1. Do not introduce unsupported facts or assumptions.
2. If the context does not contain enough information,
   clearly state that you do not have enough information.
3. Cite the source filename when providing factual claims.
4. Do not invent citations.

Context:
[Source 1: lightgbm.md]
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient t

In [433]:
HF_TOKEN = secret_value_0

print("Hugging Face token loaded:", bool(HF_TOKEN))

Hugging Face token loaded: True


In [435]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [436]:
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Transformers: 5.20.2
Accelerate: 1.15.0
GPU: Tesla T4


In [437]:
import torch
import importlib.util

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
print("bitsandbytes installed:", importlib.util.find_spec("bitsandbytes") is not None)

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
bitsandbytes installed: False


In [438]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=HF_TOKEN,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully!")
print("Device:", model.device)
print("Data type:", model.dtype)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Device: cuda:0
Data type: torch.float16


In [439]:
import torch

messages = [
    {
        "role": "system",
        "content": "You are a helpful ML engineering assistant."
    },
    {
        "role": "user",
        "content": "Explain the difference between GOSS and EFB in LightGBM."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(answer)

In LightGBM, GOSS (Gradient-Boosting with Stochastic Sampling) is an algorithm that uses stochastic sampling to improve the efficiency of gradient boosting. It works by randomly selecting a subset of samples for each iteration of the boosting process, which helps to reduce overfitting and improve generalization.
EFB (Efficient Forest Boosting) is another algorithm used in LightGBM that also uses stochastic sampling to improve the efficiency of gradient boosting. However, it differs from GOSS in that it does not use any form of regularization or early stopping during the training process. Instead, it relies on pruning weak trees to prevent overfitting and improve performance.
Overall, both GOSS and EFB are designed to improve the efficiency and performance of gradient boosting algorithms in LightGBM. The choice between them depends on the specific needs of the problem at hand, such as the size of the dataset and the desired level of accuracy.


In [443]:
import importlib
import app.generation.context_builder as context_module

importlib.reload(context_module)

build_context = context_module.build_context

context = build_context(
    ranked_results=reranked_results,
    top_k=3
)

print(context)

[Source 1: lightgbm.md]
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.

[Source 2: lightgbm.md]
# LightGBM Architecture

LightGBM is a gradient boosting framework.

[Source 3: lightgbm.md]
It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.


In [445]:
def generate_rag_answer(query, context):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an ML engineering assistant. "
                "Answer using only the provided context. "
                "Do not introduce unsupported information. "
                "If the context is insufficient, say so clearly. "
                "Cite the source filename for factual claims."
            )
        },
        {
            "role": "user",
            "content": f"""
Context:
{context}

Question:
{query}
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [446]:
answer = generate_rag_answer(query, context)
print(answer)

According to the provided sources, LightGBM uses two key techniques:

1. Gradient-based One-Side Sampling (GOSS) - This technique helps reduce the number of data instances used during training.
2. Exclusive Feature Bundling (EFB) - This method reduces the number of features by grouping mutually exclusive features together.


In [448]:
query = "What is the mathematical objective function of LightGBM?"

# Retrieve relevant chunks using the existing hybrid retriever
ranked_results = hybrid_retriever.search(
    query=query,
    query_embedding=embedding_model.encode([query])[0],
    top_k=3
)

# Build context from retrieved chunks
context = build_context(ranked_results, top_k=3)

# Generate the answer using the language model
answer = generate_rag_answer(query, context)

print("Retrieved Context:")
print(context)

print("\nGenerated Answer:")
print(answer)

Retrieved Context:
[Source 1: lightgbm.md]
# LightGBM Architecture

LightGBM is a gradient boosting framework.

[Source 2: lightgbm.md]
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.

[Source 3: lightgbm.md]
It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of data instances used during training.

Generated Answer:
The mathematical objective function of LightGBM is not explicitly stated in the given sources. However, based on the context provided:

- LightGBM is described as a gradient boosting framework.
- It uses Gradient-based One-Side Sampling (GOSS) and Exclusive Feature Bundling (EFB).
- These methods aim to reduce computational complexity and improve efficiency in training on large datasets.

While specific details about the exact objective function aren't detailed here, it can be inferred that LightGBM like

In [449]:
def generate_rag_answer(query, context):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a strictly evidence-grounded ML assistant. "
                "Use ONLY explicit facts in the provided context. "
                "Never use prior knowledge, assumptions, or speculation. "
                "If the answer is not explicitly supported by the context, "
                "respond only with: "
                "'The provided sources do not contain enough information "
                "to answer this question.' "
                "Do not add anything after this statement. "
                "Cite source filenames for factual claims."
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion:\n{query}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


query = "What is the mathematical objective function of LightGBM?"

answer = generate_rag_answer(query, context)

print(answer)

The provided sources do not contain enough information to answer this question.


In [451]:
%%writefile /kaggle/working/ml-ops-copilot/app/generation/llm_client.py

import torch


class LLMClient:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def generate(self, query, context):
        messages = [
            {
                "role": "system",
                "content": (
                    "You are a strictly evidence-grounded ML assistant. "
                    "Use ONLY explicit facts in the provided context. "
                    "Never use prior knowledge, assumptions, or speculation. "
                    "If the answer is not explicitly supported by the context, "
                    "respond only with: "
                    "'The provided sources do not contain enough information "
                    "to answer this question.' "
                    "Do not add anything after this statement. "
                    "Cite source filenames for factual claims."
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:\n{query}"
            }
        ]

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt"
        ).to(self.model.device)

        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        return self.tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )

Overwriting /kaggle/working/ml-ops-copilot/app/generation/llm_client.py


In [452]:
import sys
import importlib

sys.path.insert(0, "/kaggle/working/ml-ops-copilot")

from app.generation.llm_client import LLMClient

import app.generation.llm_client as llm_module

importlib.reload(llm_module)

llm_client = llm_module.LLMClient(
    model=model,
    tokenizer=tokenizer
)

answer = llm_client.generate(
    query=query,
    context=context
)

print(answer)

The provided sources do not contain enough information to answer this question.


In [453]:
import os

project_root = "/kaggle/working/ml-ops-copilot"

for root, dirs, files in os.walk(project_root):
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(project_root, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in sorted(files):
        print(f"{indent}    {file}")

ml-ops-copilot/
    configs/
    tests/
    data/
        sample/
            lightgbm.md
    docs/
    app/
        main.py
        core/
            __init__.py
            document_store.py
        ingestion/
            __init__.py
            document_loader.py
            text_chunker.py
        retrieval/
            __init__.py
            bm25.py
            embeddings.py
            hybrid_search.py
            reranker.py
            vector_store.py
        evaluation/
        generation/
            context_builder.py
            llm_client.py
            prompt_builder.py
        api/


In [454]:
%%writefile /kaggle/working/ml-ops-copilot/app/generation/rag_pipeline.py

from app.generation.context_builder import build_context


class RAGPipeline:
    def __init__(
        self,
        embedding_model,
        hybrid_retriever,
        reranker,
        llm_client,
        retrieval_top_k=5,
        context_top_k=3
    ):
        self.embedding_model = embedding_model
        self.hybrid_retriever = hybrid_retriever
        self.reranker = reranker
        self.llm_client = llm_client
        self.retrieval_top_k = retrieval_top_k
        self.context_top_k = context_top_k

    def run(self, query):

        # Step 1: Generate query embedding
        query_embedding = self.embedding_model.encode([query])[0]

        # Step 2: Retrieve relevant chunks using hybrid search
        retrieved_results = self.hybrid_retriever.search(
            query=query,
            query_embedding=query_embedding,
            top_k=self.retrieval_top_k
        )

        # Step 3: Rerank retrieved chunks
        reranked_results = self.reranker.rerank(
            query=query,
            results=retrieved_results,
            top_k=self.context_top_k
        )

        # Step 4: Build context
        context = build_context(
            reranked_results,
            top_k=self.context_top_k
        )

        # Step 5: Generate grounded answer
        answer = self.llm_client.generate(
            query=query,
            context=context
        )

        # Step 6: Collect source information
        sources = [
            {
                "filename": result["chunk"]["filename"],
                "chunk_id": result["chunk"]["chunk_id"],
                "score": result["score"]
            }
            for result in reranked_results
        ]

        return {
            "query": query,
            "answer": answer,
            "context": context,
            "sources": sources
        }

Writing /kaggle/working/ml-ops-copilot/app/generation/rag_pipeline.py


In [455]:
import sys
import importlib

sys.path.insert(0, "/kaggle/working/ml-ops-copilot")

from app.retrieval.embeddings import EmbeddingModel
from app.retrieval.hybrid_search import HybridRetriever
from app.retrieval.reranker import CrossEncoderReranker
from app.generation.llm_client import LLMClient
from app.generation.rag_pipeline import RAGPipeline

print("All modules imported successfully.")

All modules imported successfully.


In [456]:
# Initialize the RAG pipeline

pipeline = RAGPipeline(
    embedding_model=embedding_model,
    hybrid_retriever=hybrid_retriever,
    reranker=reranker,
    llm_client=llm_client,
    retrieval_top_k=3,
    context_top_k=3
)

print("RAG Pipeline initialized successfully.")

RAG Pipeline initialized successfully.


In [457]:
result = pipeline.run(
    "What are the two key techniques used by LightGBM?"
)

print("QUESTION:")
print(result["query"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

print("\nCONTEXT:")
print(result["context"])

QUESTION:
What are the two key techniques used by LightGBM?

ANSWER:
According to Source 3 from "lightgbm.md", the two key techniques used by LightGBM are:

1. Gradient-based One-Side Sampling (GOSS)
2. Exclusive Feature Bundling (EFB)

SOURCES:
{'filename': 'lightgbm.md', 'chunk_id': '22bbe056-23e0-4f94-8e51-e7609e9ec798', 'score': 0.6620275378227234}
{'filename': 'lightgbm.md', 'chunk_id': '7f49b480-97b6-478b-9421-4bb48c785dab', 'score': -0.28183266520500183}
{'filename': 'lightgbm.md', 'chunk_id': '6eef6c80-983c-490e-bffa-4f44a9074a72', 'score': -2.208390474319458}

CONTEXT:
[Source 1: lightgbm.md]
EFB reduces the number of features by bundling mutually exclusive features. LightGBM is designed for efficient training on large datasets.

[Source 2: lightgbm.md]
# LightGBM Architecture

LightGBM is a gradient boosting framework.

[Source 3: lightgbm.md]
It uses two important techniques:
Gradient-based One-Side Sampling (GOSS)
Exclusive Feature Bundling (EFB)

GOSS reduces the number of

In [458]:
result = pipeline.run(
    "What is the mathematical objective function of LightGBM?"
)

print("QUESTION:")
print(result["query"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

QUESTION:
What is the mathematical objective function of LightGBM?

ANSWER:
The provided sources do not contain enough information to answer this question.

SOURCES:
{'filename': 'lightgbm.md', 'chunk_id': '22bbe056-23e0-4f94-8e51-e7609e9ec798', 'score': 0.3780233561992645}
{'filename': 'lightgbm.md', 'chunk_id': '7f49b480-97b6-478b-9421-4bb48c785dab', 'score': 0.1604820340871811}
{'filename': 'lightgbm.md', 'chunk_id': '6eef6c80-983c-490e-bffa-4f44a9074a72', 'score': -11.380264282226562}


In [459]:
%%writefile /kaggle/working/ml-ops-copilot/app/evaluation/evaluator.py

class RAGEvaluator:

    def evaluate_answer(self, answer, expected_keywords):
        answer_lower = answer.lower()

        matched_keywords = [
            keyword
            for keyword in expected_keywords
            if keyword.lower() in answer_lower
        ]

        score = (
            len(matched_keywords) / len(expected_keywords)
            if expected_keywords else 0.0
        )

        return {
            "score": score,
            "matched_keywords": matched_keywords,
            "missing_keywords": [
                keyword
                for keyword in expected_keywords
                if keyword not in matched_keywords
            ],
            "passed": score == 1.0
        }

    def evaluate_abstention(self, answer):
        expected_phrase = (
            "do not contain enough information"
        )

        return {
            "passed": expected_phrase in answer.lower()
        }

Writing /kaggle/working/ml-ops-copilot/app/evaluation/evaluator.py


In [460]:
import sys
import importlib

sys.path.insert(0, "/kaggle/working/ml-ops-copilot")

import app.evaluation.evaluator as evaluator_module

importlib.reload(evaluator_module)

RAGEvaluator = evaluator_module.RAGEvaluator
evaluator = RAGEvaluator()

# Test 1: Answerable question
supported_result = pipeline.run(
    "What are the two key techniques used by LightGBM?"
)

supported_evaluation = evaluator.evaluate_answer(
    answer=supported_result["answer"],
    expected_keywords=[
        "Gradient-based One-Side Sampling (GOSS)",
        "Exclusive Feature Bundling (EFB)"
    ]
)

# Test 2: Unanswerable question
unsupported_result = pipeline.run(
    "What is the mathematical objective function of LightGBM?"
)

abstention_evaluation = evaluator.evaluate_abstention(
    unsupported_result["answer"]
)

print("TEST 1 — Answer Evaluation")
print(supported_evaluation)

print("\nTEST 2 — Abstention Evaluation")
print(abstention_evaluation)

print("\nOVERALL:")
print(
    "PASSED"
    if supported_evaluation["passed"]
    and abstention_evaluation["passed"]
    else "FAILED"
)

TEST 1 — Answer Evaluation
{'score': 1.0, 'matched_keywords': ['Gradient-based One-Side Sampling (GOSS)', 'Exclusive Feature Bundling (EFB)'], 'missing_keywords': [], 'passed': True}

TEST 2 — Abstention Evaluation
{'passed': True}

OVERALL:
PASSED


In [461]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

files_to_check = [
    "app/main.py",
    "app/api",
    "requirements.txt",
    "pyproject.toml",
    "Dockerfile"
]

for item in files_to_check:
    path = root / item
    print(f"\n--- {item} ---")

    if path.is_file():
        print(path.read_text()[:10000])
    elif path.is_dir():
        for file in sorted(path.rglob("*")):
            if file.is_file():
                print(file.relative_to(root))
    else:
        print("Not found")


--- app/main.py ---
def main():
    print("ML Ops Copilot is starting...")


if __name__ == "__main__":
    main()

--- app/api ---

--- requirements.txt ---
Not found

--- pyproject.toml ---
Not found

--- Dockerfile ---
Not found


In [462]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

files = [
    "app/core/document_store.py",
    "app/ingestion/document_loader.py",
    "app/ingestion/text_chunker.py",
    "app/retrieval/embeddings.py",
    "app/retrieval/vector_store.py",
    "app/retrieval/bm25.py",
    "app/retrieval/hybrid_search.py",
    "app/retrieval/reranker.py",
    "app/generation/context_builder.py",
    "app/generation/llm_client.py",
    "app/generation/rag_pipeline.py",
]

for filename in files:
    path = root / filename

    print(f"\n{'=' * 60}")
    print(filename)
    print('=' * 60)

    if path.exists():
        print(path.read_text())
    else:
        print("FILE NOT FOUND")


app/core/document_store.py

from uuid import uuid4


class DocumentStore:
    def __init__(self):
        self.documents = {}

    def create(self, document_data):
        document_id = str(uuid4())

        document = {
            "id": document_id,
            **document_data
        }

        self.documents[document_id] = document
        return document

    def get(self, document_id):
        return self.documents.get(document_id)


app/ingestion/document_loader.py

from pathlib import Path
from pypdf import PdfReader


def load_document(file_path):
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"File not found: {path}"
        )

    extension = path.suffix.lower()

    if extension in [".md", ".txt"]:
        text = path.read_text(encoding="utf-8")
        page_count = None

    elif extension == ".pdf":
        reader = PdfReader(str(path))

        pages = []

        for page_number, page in enumerate(reader.pages, start

In [463]:
%%writefile /kaggle/working/ml-ops-copilot/app/core/rag_factory.py

import os
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from app.ingestion.document_loader import load_document
from app.ingestion.text_chunker import sentence_aware_chunking

from app.retrieval.embeddings import EmbeddingModel
from app.retrieval.vector_store import VectorStore
from app.retrieval.bm25 import BM25Retriever
from app.retrieval.hybrid_search import HybridRetriever
from app.retrieval.reranker import CrossEncoderReranker

from app.generation.llm_client import LLMClient
from app.generation.rag_pipeline import RAGPipeline


def create_rag_pipeline(model=None, tokenizer=None):

    project_root = Path(__file__).resolve().parents[2]

    # 1. Load sample documents
    sample_path = project_root / "data" / "sample" / "lightgbm.md"

    document = load_document(sample_path)

    chunks = sentence_aware_chunking(
        text=document["content"],
        document_id="sample-lightgbm",
        filename=document["filename"],
        chunk_size=500
    )

    if not chunks:
        raise ValueError("No chunks were created from the sample document.")

    # 2. Initialize embedding model
    embedding_model = EmbeddingModel()

    embeddings = embedding_model.encode(
        [chunk["text"] for chunk in chunks]
    )

    # 3. Initialize vector store
    vector_store = VectorStore(
        vector_size=embeddings.shape[1]
    )

    vector_store.add_chunks(chunks, embeddings)

    # 4. Initialize sparse retrieval
    bm25_retriever = BM25Retriever(chunks)

    # 5. Initialize hybrid retrieval
    hybrid_retriever = HybridRetriever(
        vector_store=vector_store,
        bm25_retriever=bm25_retriever
    )

    # 6. Initialize reranker
    reranker = CrossEncoderReranker()

    # 7. Load language model if not supplied
    if model is None or tokenizer is None:

        model_name = os.getenv(
            "LLM_MODEL_NAME",
            "Qwen/Qwen2.5-1.5B-Instruct"
        )

        hf_token = os.getenv("HF_TOKEN")

        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            token=hf_token
        )

        model_dtype = (
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            token=hf_token,
            dtype=model_dtype,
            device_map="auto"
        )

        model.eval()

    # 8. Initialize LLM client
    llm_client = LLMClient(
        model=model,
        tokenizer=tokenizer
    )

    # 9. Assemble complete RAG pipeline
    return RAGPipeline(
        embedding_model=embedding_model,
        hybrid_retriever=hybrid_retriever,
        reranker=reranker,
        llm_client=llm_client,
        retrieval_top_k=5,
        context_top_k=3
    )

Writing /kaggle/working/ml-ops-copilot/app/core/rag_factory.py


In [464]:
import sys
sys.path.insert(0, "/kaggle/working/ml-ops-copilot")

from app.core.rag_factory import create_rag_pipeline

api_pipeline = create_rag_pipeline(
    model=model,
    tokenizer=tokenizer
)

test_result = api_pipeline.run(
    "What are the two key techniques used by LightGBM?"
)

print("ANSWER:")
print(test_result["answer"])
print("\nSOURCES:")
print(test_result["sources"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

ANSWER:
The two key techniques used by LightGBM are Gradient-based One-Side Sampling (GOSS) and Exclusive Feature Bundling (EFB).

SOURCES:
[{'filename': 'lightgbm.md', 'chunk_id': 'c7aabd6f-fb96-4272-b6a2-a0dc90516d16', 'score': 7.777410507202148}]


In [465]:
%%writefile /kaggle/working/ml-ops-copilot/app/main.py

from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from app.core.rag_factory import create_rag_pipeline


class QueryRequest(BaseModel):
    query: str = Field(..., min_length=3, description="User question")


class QueryResponse(BaseModel):
    query: str
    answer: str
    sources: list[dict]


def create_app(pipeline=None):

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        if app.state.pipeline is None:
            app.state.pipeline = create_rag_pipeline()
        yield

    app = FastAPI(
        title="ML Ops Copilot",
        description="Evidence-grounded RAG API for ML engineering documentation.",
        version="1.0.0",
        lifespan=lifespan
    )

    app.state.pipeline = pipeline

    @app.get("/health")
    def health():
        return {
            "status": "healthy",
            "pipeline_ready": app.state.pipeline is not None
        }

    @app.post("/query", response_model=QueryResponse)
    def query_documents(request: QueryRequest):
        if app.state.pipeline is None:
            raise HTTPException(
                status_code=503,
                detail="RAG pipeline is not ready."
            )

        try:
            result = app.state.pipeline.run(request.query)
            return {
                "query": result["query"],
                "answer": result["answer"],
                "sources": result["sources"]
            }
        except Exception as error:
            raise HTTPException(
                status_code=500,
                detail=str(error)
            )

    return app


app = create_app()

Overwriting /kaggle/working/ml-ops-copilot/app/main.py


In [466]:
from fastapi.testclient import TestClient
from app.main import create_app

test_app = create_app(pipeline=api_pipeline)

with TestClient(test_app) as client:

    health_response = client.get("/health")
    print("HEALTH:", health_response.status_code)
    print(health_response.json())

    query_response = client.post(
        "/query",
        json={"query": "What are the two key techniques used by LightGBM?"}
    )

    print("\nQUERY:", query_response.status_code)
    print(query_response.json())

HEALTH: 200
{'status': 'healthy', 'pipeline_ready': True}

QUERY: 200
{'query': 'What are the two key techniques used by LightGBM?', 'answer': 'The two key techniques used by LightGBM are Gradient-based One-Side Sampling (GOSS) and Exclusive Feature Bundling (EFB).', 'sources': [{'filename': 'lightgbm.md', 'chunk_id': 'c7aabd6f-fb96-4272-b6a2-a0dc90516d16', 'score': 7.777410507202148}]}


In [467]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

requirements = """
fastapi>=0.115.0
uvicorn[standard]>=0.30.0
pydantic>=2.0.0
torch>=2.2.0
transformers>=5.0.0
accelerate>=1.0.0
sentence-transformers>=3.0.0
qdrant-client>=1.12.0
rank-bm25>=0.2.2
pypdf>=5.0.0
"""

dockerfile = """
FROM python:3.11-slim

WORKDIR /app

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1
ENV HF_HOME=/app/.cache/huggingface

COPY requirements.txt .

RUN pip install --no-cache-dir --upgrade pip \
    && pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

dockerignore = """
.git
.github
__pycache__
*.pyc
.ipynb_checkpoints
.env
.venv
venv
data/raw
data/processed
"""

(root / "requirements.txt").write_text(requirements.strip() + "\n")
(root / "Dockerfile").write_text(dockerfile.strip() + "\n")
(root / ".dockerignore").write_text(dockerignore.strip() + "\n")

print("Created:")
for filename in ["requirements.txt", "Dockerfile", ".dockerignore"]:
    print(f"- {filename}: {(root / filename).exists()}")

Created:
- requirements.txt: True
- Dockerfile: True
- .dockerignore: True


In [468]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

packages = [
    "app",
    "app/core",
    "app/ingestion",
    "app/retrieval",
    "app/generation",
    "app/evaluation",
    "app/api",
]

for package in packages:
    init_file = root / package / "__init__.py"
    init_file.parent.mkdir(parents=True, exist_ok=True)

    if not init_file.exists():
        init_file.write_text("")

    print(f"{package}: OK")

app: OK
app/core: OK
app/ingestion: OK
app/retrieval: OK
app/generation: OK
app/evaluation: OK
app/api: OK


In [469]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")
test_file = root / "tests" / "test_api.py"

test_code = '''
from fastapi.testclient import TestClient

from app.main import create_app


class FakePipeline:
    def run(self, query):
        return {
            "query": query,
            "answer": "GOSS and EFB.",
            "context": "Test context",
            "sources": [
                {
                    "filename": "lightgbm.md",
                    "chunk_id": "test-chunk",
                    "score": 0.9
                }
            ]
        }


def test_health_endpoint():
    app = create_app(pipeline=FakePipeline())

    with TestClient(app) as client:
        response = client.get("/health")

    assert response.status_code == 200
    assert response.json()["status"] == "healthy"
    assert response.json()["pipeline_ready"] is True


def test_query_endpoint():
    app = create_app(pipeline=FakePipeline())

    with TestClient(app) as client:
        response = client.post(
            "/query",
            json={"query": "What are GOSS and EFB?"}
        )

    assert response.status_code == 200
    data = response.json()
    assert data["answer"] == "GOSS and EFB."
    assert data["sources"][0]["filename"] == "lightgbm.md"


def test_query_rejects_short_question():
    app = create_app(pipeline=FakePipeline())

    with TestClient(app) as client:
        response = client.post(
            "/query",
            json={"query": "Hi"}
        )

    assert response.status_code == 422
'''

test_file.write_text(test_code.strip() + "\n")

print("Created:", test_file)

Created: /kaggle/working/ml-ops-copilot/tests/test_api.py


In [470]:
!cd /kaggle/working/ml-ops-copilot && pytest -q


==================================== ERRORS ====================================
______________________ ERROR collecting tests/test_api.py ______________________
ImportError while importing test module '/kaggle/working/ml-ops-copilot/tests/test_api.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.13/importlib/__init__.py:88: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_api.py:3: in <module>
    from app.main import create_app
E   ModuleNotFoundError: No module named 'app'
=========================== short test summary info ============================
ERROR tests/test_api.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 0.28s


In [471]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")
(root / "pytest.ini").write_text(
    "[pytest]\n"
    "pythonpath = .\n"
    "testpaths = tests\n"
)

print("pytest.ini created successfully.")

pytest.ini created successfully.


In [472]:
!cd /kaggle/working/ml-ops-copilot && python -m pytest -q

...                                                                      [100%]
3 passed in 10.27s


In [473]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

for path in sorted(root.rglob("*")):
    if path.is_file() and ".git" not in path.parts:
        print(path.relative_to(root))

.dockerignore
.pytest_cache/.gitignore
.pytest_cache/CACHEDIR.TAG
.pytest_cache/README.md
.pytest_cache/v/cache/lastfailed
.pytest_cache/v/cache/nodeids
Dockerfile
app/__init__.py
app/__pycache__/__init__.cpython-313.pyc
app/__pycache__/main.cpython-313.pyc
app/api/__init__.py
app/core/__init__.py
app/core/__pycache__/__init__.cpython-313.pyc
app/core/__pycache__/document_store.cpython-313.pyc
app/core/__pycache__/rag_factory.cpython-313.pyc
app/core/document_store.py
app/core/rag_factory.py
app/evaluation/__init__.py
app/evaluation/__pycache__/evaluator.cpython-313.pyc
app/evaluation/evaluator.py
app/generation/__init__.py
app/generation/__pycache__/__init__.cpython-313.pyc
app/generation/__pycache__/context_builder.cpython-313.pyc
app/generation/__pycache__/llm_client.cpython-313.pyc
app/generation/__pycache__/prompt_builder.cpython-313.pyc
app/generation/__pycache__/rag_pipeline.cpython-313.pyc
app/generation/context_builder.py
app/generation/llm_client.py
app/generation/prompt_buil

In [474]:
from pathlib import Path
import shutil

root = Path("/kaggle/working/ml-ops-copilot")

for name in ["__pycache__", ".pytest_cache"]:
    for path in list(root.rglob(name)):
        if path.is_dir():
            shutil.rmtree(path)

print("Remaining cache directories:")
for path in root.rglob("*"):
    if path.is_dir() and path.name in ["__pycache__", ".pytest_cache"]:
        print(path)

print("\n.gitignore exists:", (root / ".gitignore").exists())

Remaining cache directories:

.gitignore exists: False


In [475]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

gitignore = """
__pycache__/
*.py[cod]
.pytest_cache/
.venv/
venv/
.env
*.log
.ipynb_checkpoints/
.cache/
"""

dockerignore = """
.git
.github
__pycache__
*.pyc
.pytest_cache
.ipynb_checkpoints
.env
.venv
venv
.cache
data/raw
data/processed
"""

(root / ".gitignore").write_text(gitignore.strip() + "\n")
(root / ".dockerignore").write_text(dockerignore.strip() + "\n")

print(".gitignore:", (root / ".gitignore").exists())
print(".dockerignore:", (root / ".dockerignore").exists())

.gitignore: True
.dockerignore: True


In [479]:
from pathlib import Path

p = Path("/kaggle/working/ml-ops-copilot/README.md")

content = [
    "# ML Ops Copilot",
    "## Evidence-Grounded RAG Assistant",
    "",
    "A RAG application for answering machine learning documentation questions.",
    "",
    "### Features",
    "- PDF, TXT, Markdown ingestion",
    "- Sentence-aware chunking",
    "- Sentence Transformers embeddings",
    "- Qdrant dense retrieval",
    "- BM25 sparse retrieval",
    "- Reciprocal Rank Fusion",
    "- Cross-Encoder reranking",
    "- Qwen2.5-1.5B generation",
    "- Evidence-grounded answers",
    "- Source attribution",
    "- Abstention for unsupported questions",
    "- FastAPI and Docker",
    "- Automated pytest tests",
    "",
    "### Architecture",
    "Documents -> Chunking -> Embeddings -> Dense and Sparse Retrieval",
    "-> Hybrid Search -> Reranking -> Context -> LLM -> Answer",
    "",
    "### Run",
    "```bash",
    "pip install -r requirements.txt",
    "uvicorn app.main:app --host 0.0.0.0 --port 8000",
    "```",
    "",
    "API documentation: http://localhost:8000/docs",
    "",
    "### Tests",
    "```bash",
    "python -m pytest -q",
    "```",
    "",
    "Verified result: 3 passed.",
    "",
    "### Limitations",
    "Portfolio prototype, not a production deployment.",
    "Qdrant uses in-memory storage. Documents are not persisted.",
    "The sample knowledge base currently contains LightGBM documentation.",
    "Evaluation uses basic rule-based checks.",
    "",
    "### Author",
    "Rasul Aramian",
    "",
    "GitHub: https://github.com/RasulAramian",
    "LinkedIn: https://www.linkedin.com/in/rasul-aramian-958010115/"
]

p.write_text("\n".join(content) + "\n", encoding="utf-8")

print("README created:", p.exists())
print("Size:", p.stat().st_size)

README created: True
Size: 1221


In [480]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

required = [
    "README.md",
    "requirements.txt",
    "Dockerfile",
    ".dockerignore",
    ".gitignore",
    "pytest.ini",
    "app/main.py",
    "app/core/rag_factory.py",
    "app/generation/rag_pipeline.py",
    "tests/test_api.py",
    "data/sample/lightgbm.md"
]

for name in required:
    print(("OK      " if (root / name).is_file() else "MISSING ") + name)

readme_text = (root / "README.md").read_text(encoding="utf-8")
print("\nREADME contains project title:", "# ML Ops Copilot" in readme_text)
print("README contains limitations:", "Limitations" in readme_text)

print("\nPotential secret files:")
for name in [".env", "credentials.json", "token.txt"]:
    if (root / name).exists():
        print("CHECK:", name)
print("Check complete.")

OK      README.md
OK      requirements.txt
OK      Dockerfile
OK      .dockerignore
OK      .gitignore
OK      pytest.ini
OK      app/main.py
OK      app/core/rag_factory.py
OK      app/generation/rag_pipeline.py
OK      tests/test_api.py
OK      data/sample/lightgbm.md

README contains project title: True
README contains limitations: True

Potential secret files:
Check complete.


In [482]:
from pathlib import Path

root = Path("/kaggle/working/ml-ops-copilot")

requirements = [
    "fastapi>=0.115.0",
    "uvicorn[standard]>=0.30.0",
    "pydantic>=2.0.0",
    "torch>=2.2.0",
    "transformers>=5.0.0",
    "accelerate>=1.0.0",
    "sentence-transformers>=3.0.0",
    "qdrant-client>=1.12.0",
    "rank-bm25>=0.2.2",
    "pypdf>=5.0.0",
    "httpx>=0.27.0",
    "pytest>=8.0.0"
]

(root / "requirements.txt").write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

print("Updated requirements.txt")

Updated requirements.txt


In [483]:
import os
import subprocess
import base64
from kaggle_secrets import UserSecretsClient

# 1. Load GitHub token securely
secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")

# 2. Set project directory
PROJECT_DIR = "/kaggle/working/ml-ops-copilot"
os.chdir(PROJECT_DIR)

# 3. Run Git commands safely
def run_git(command):
    result = subprocess.run(
        ["git"] + command,
        capture_output=True,
        text=True,
        env=git_env
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

# 4. Configure temporary authentication (token is never printed)
auth = base64.b64encode(
    f"x-access-token:{github_token}".encode()
).decode()

git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
git_env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"
git_env["GIT_TERMINAL_PROMPT"] = "0"

# 5. Initialize Git
if not os.path.isdir(".git"):
    run_git(["init"])

run_git(["branch", "-M", "main"])
run_git(["config", "user.name", "Rasul Aramian"])
run_git(["config", "user.email", "rasularamian@users.noreply.github.com"])

# 6. Connect GitHub repository
REPO_URL = "https://github.com/RasulAramian/ml-ops-copilot.git"

try:
    run_git(["remote", "set-url", "origin", REPO_URL])
except RuntimeError:
    run_git(["remote", "add", "origin", REPO_URL])

# 7. Stage project files
run_git(["add", "."])

# 8. Prevent accidental upload of obvious secret files
staged = run_git(["diff", "--cached", "--name-only"]).splitlines()
sensitive = [
    f for f in staged
    if os.path.basename(f).lower() in {
        ".env", "credentials.json", "secrets.json"
    }
]

if sensitive:
    raise RuntimeError(f"Potential secret files detected: {sensitive}")

# 9. Commit
if staged:
    run_git(["commit", "-m", "Initial release: ML Ops Copilot"])
else:
    print("No new files to commit.")

# 10. Push to GitHub
run_git(["push", "-u", "origin", "main"])

print("SUCCESS: Project pushed to GitHub.")
print("Repository: " + REPO_URL)

SUCCESS: Project pushed to GitHub.
Repository: https://github.com/RasulAramian/ml-ops-copilot.git
